# Streaming scoring demo

The streaming demo runs from the CLI, not from this notebook (a notebook kernel and `awaitTermination` do not mix well). This notebook documents the flow and inspects the scored output afterwards.

**Prerequisites:** a trained model (`make train`). Then, from the repository root, start the scoring job in one terminal:

```bash
poetry run transaction-risk score-stream \
  --input-stream data/streaming/incoming \
  --model models/fraud_risk_pipeline \
  --output data/streaming/scored \
  --checkpoint data/streaming/checkpoints/scoring
```

and drop PaySim-style CSV files into `data/streaming/incoming/` from another terminal. Each micro-batch is scored with the same feature pipeline used in batch mode and appended to `data/streaming/scored/`.

In [ ]:
from transaction_risk.spark.session import create_spark_session_from_yaml
spark = create_spark_session_from_yaml('../conf/spark.local.yaml')


In [ ]:
# Inspect scored streaming output after the job has processed at least one file
from pyspark.sql import functions as F

scored = spark.read.parquet('../data/streaming/scored')
scored.select('step', 'type', 'amount', 'fraud_probability', 'is_alert', 'stream_batch_id').show(10)
scored.groupBy('stream_batch_id').agg(
    F.count(F.lit(1)).alias('transactions'),
    F.sum('is_alert').alias('alerts'),
).orderBy('stream_batch_id').show()
spark.stop()